# Small + rich circuit scaling sweep

The 10-wire small circuit (target wire 9) sits beside the 100-wire depth-8 rich circuit (`task="concat"`). The input has 10 + 100 bits and the output has 1 + 100 logits. Small-circuit inputs come only from its 80% train pool, while rich-circuit inputs are always fresh. Training length and LR recipe match `rich_sweep.ipynb`. **The metric is held-out loss on the small output only** (output 0): held-out small inputs, each paired with fresh rich inputs.

`SMALL_WEIGHT = None` averages the loss over all 101 outputs. A number w in [0, 1] instead uses w · small + (1 − w) · mean(rich).

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

import jax
print(jax.devices())

In [ ]:
import json
from pathlib import Path

import numpy as np

from train import RunConfig, run

N_WIRES, CIRC_DEPTH, CIRCUIT_SEED = 100, 8, 0                  # rich circuit
SMALL_WIRES, SMALL_DEPTH, SMALL_SEED, SMALL_WIRE = 10, 4, 0, 9  # small circuit
TRAIN_FRAC = 0.8
SMALL_WEIGHT = None
GRID = [(32, 2), (48, 2), (64, 3), (96, 3), (128, 4),
        (180, 5), (256, 6), (360, 7), (512, 8)]
LR_GRID = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
TUNE_STEPS, FULL_STEPS, BATCH = 5_000, 50_000, 256
OUT_DIR = f"runs/concat_c{N_WIRES}x{CIRC_DEPTH}"
LR_TABLE = Path(OUT_DIR) / f"lr_table_sw{SMALL_WEIGHT}.json"


def cfg(width, depth, lr, steps, out_dir, seed=0):
    return RunConfig(width=width, mlp_depth=depth, lr=lr, steps=steps, batch=BATCH,
                     task="concat", n_wires=N_WIRES, circ_depth=CIRC_DEPTH,
                     circuit_seed=CIRCUIT_SEED, small_wires=SMALL_WIRES,
                     small_depth=SMALL_DEPTH, small_seed=SMALL_SEED,
                     small_wire=SMALL_WIRE, small_weight=SMALL_WEIGHT,
                     train_frac=TRAIN_FRAC, data_order="epoch",
                     model_seed=seed, out_dir=out_dir)


def small_ho(c):
    """Held-out small-circuit loss curve (output 0) of a finished run."""
    r = np.load(c.npz_path)
    return r["eval_steps"], r["per_out_loss_ho"][:, 0]

## LR sweep (on held-out small-circuit loss)

In [ ]:
table = {}
for w, d in GRID:
    losses = {}
    for lr in LR_GRID:
        c = cfg(w, d, lr, TUNE_STEPS, f"{OUT_DIR}/tune")
        run(c)
        losses[lr] = float(small_ho(c)[1][-1])
    table[f"w{w}d{d}"] = best = min(losses, key=losses.get)
    edge = "  <- grid edge" if best in (LR_GRID[0], LR_GRID[-1]) else ""
    print(f"w{w}d{d}: best lr {best:g}{edge}")
LR_TABLE.write_text(json.dumps(table, indent=2))

## Main runs

In [ ]:
table = json.loads(LR_TABLE.read_text())
for w, d in GRID:
    run(cfg(w, d, table[f"w{w}d{d}"], FULL_STEPS, OUT_DIR))

## Results

In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from train import load_run


def n_params(w, d):
    return d * (w + 8 * w * w) + w  # non-embedding params


def fit_plot(ax, N, L):
    """Scatter L vs N with a fitted L = E + A N^-alpha."""
    N, L = np.array(N), np.array(L)
    ax.plot(N, L, "o")
    f = lambda lN, lE, lA, a: np.log(np.exp(lE) + np.exp(lA - a * lN))
    try:
        (lE, lA, a), _ = curve_fit(f, np.log(N), np.log(L),
                                   p0=(np.log(0.9 * L.min()), 1.0, 0.3), maxfev=20000)
        xs = np.logspace(np.log10(N.min()), np.log10(N.max()), 100)
        ax.plot(xs, np.exp(lE) + np.exp(lA) * xs ** -a, "k--",
                label=f"L = {np.exp(lE):.3f} + {np.exp(lA):.3g} N^-{a:.2f}")
        ax.legend(fontsize=8)
    except (RuntimeError, ValueError, TypeError):
        print("fit failed (too few points?)")
    ax.set(xscale="log", yscale="log", xlabel="non-embedding params N")


N, L = [], []
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
for (w, d), col in zip(GRID, plt.cm.viridis(np.linspace(0, 0.9, len(GRID)))):
    c = cfg(w, d, table[f"w{w}d{d}"], FULL_STEPS, OUT_DIR)
    if not c.npz_path.exists():
        continue
    steps, loss = small_ho(c)
    m = steps > 0
    ax1.plot(steps[m] * BATCH, loss[m], color=col, label=f"w{w}d{d}")
    N.append(n_params(w, d)); L.append(loss[-1])
fit_plot(ax2, N, L)
for ax in (ax1, ax2):
    ax.axhline(np.log(2), color="gray", ls=":", lw=0.8)
ax1.set(xscale="log", yscale="log", xlabel="samples D", ylabel="held-out small-circuit BCE",
        title="L(D)")
ax2.set(ylabel="final held-out small-circuit BCE", title=f"L(N) at {FULL_STEPS:,} steps")
ax1.legend(fontsize=7)
plt.tight_layout()